# z3 — 30-minute v3 smoke run on a T4

**Runtime → Change runtime type → T4 GPU** before running.

This notebook is a **runner**, not a result. It runs `z3/train_v3.py` for
30 minutes on the mixed NaNROUTER diverse + Gutenberg corpus:

- `corpora-paragraphs-0010` (`paragraphs-0010`)
- `corpora-short-stories-0004` (`short_stories-0004`)
- `corpora-sentences-0010` (`sentences-0010`)
- `hw5-gutenberg-c256-g1` (21-book Gutenberg reference)

The T4 profile requested when the trainer exposes the flags is `d_model=256`,
8 layers, 8 heads, `ctx=1024`, 4 MoE layers, 32 routed experts top-2 + 2
shared, and MTP=2. The 384-wide / 12-layer v3 default also fits a T4. An 8k
context does **not** fit a T4; see the final cell.

Artifacts are written to `/content/drive/MyDrive/nanlmv3_v3`.


In [ ]:
!nvidia-smi

import torch

assert torch.cuda.is_available(), (
    'No CUDA GPU. Runtime -> Change runtime type -> T4 GPU, then reconnect.'
)
props = torch.cuda.get_device_properties(0)
name = torch.cuda.get_device_name(0)
print()
print(f'GPU: {name}')
print(f'capability: {torch.cuda.get_device_capability(0)}')
print(f'memory: {props.total_memory / 1e9:.1f} GB')
print(f'torch: {torch.__version__} | cuda: {torch.version.cuda}')
if 'T4' not in name:
    print('WARNING: this notebook is sized for a T4 (16 GB). Other GPUs work but timings differ.')


In [ ]:
!pip install -q tokenizers zstandard


## 1. Clone or update the repositories

The notebook refuses to continue if `z3/train_v3.py` is absent. This
notebook prepares the corpus and tokenizer; the training script owns the
training loop, train/validation split, and exact artifact names.


In [ ]:
import os
import pathlib
import subprocess

REPOS = {
    '/content/NaNROUTER': 'https://github.com/shemshallah/NaNROUTER.git',
    '/content/nanlm_experimental': 'https://github.com/shemshallah/nanlm_experimental.git',
}

def sync_repo(path, url):
    path = pathlib.Path(path)
    if not path.exists():
        print('cloning', url)
        subprocess.run(['git', 'clone', '--depth', '1', url, str(path)], check=True)
        return
    if not (path / '.git').exists():
        raise RuntimeError(f'{path} exists but is not a git checkout; move it aside first')
    print('updating', path)
    env = dict(os.environ, GIT_TERMINAL_PROMPT='0')
    subprocess.run(
        ['git', '-C', str(path), 'fetch', '--depth', '1', 'origin'],
        check=True, env=env,
    )
    pull = subprocess.run(
        ['git', '-C', str(path), 'pull', '--ff-only'],
        text=True, capture_output=True, env=env,
    )
    if pull.returncode != 0:
        print('WARNING: could not fast-forward; keeping the existing checkout')
        print(pull.stderr.strip())

for path, url in REPOS.items():
    sync_repo(path, url)

EXP = pathlib.Path('/content/nanlm_experimental')
TRAIN = EXP / 'z3' / 'train_v3.py'
assert (EXP / 'z3' / 'v3_model.py').exists(), 'missing z3/v3_model.py after update'
if not TRAIN.exists():
    raise SystemExit(
        'missing z3/train_v3.py; pull the commit that adds the trainer. '
        'This notebook only runs the trainer.'
    )
print('trainer:', TRAIN)


## 2. Mount Drive at the original `/content/drive`

If Colab says the mountpoint already contains files, the cell does not
delete anything. It either recognizes an existing `MyDrive` or moves
non-Drive leftovers to `/content/drive_pre_mount_backup` and retries.


In [ ]:
from google.colab import drive
import os
import shutil
import time

MOUNT = '/content/drive'

def mount_drive():
    try:
        drive.mount(MOUNT, force_remount=True)
    except Exception as exc:
        message = str(exc)
        if 'mountpoint must not already contain files' not in message.lower():
            raise
        if os.path.isdir(os.path.join(MOUNT, 'MyDrive')):
            print('Drive already mounted at', MOUNT, '- leaving MyDrive untouched')
            return
        backup = '/content/drive_pre_mount_backup'
        os.makedirs(backup, exist_ok=True)
        moved = []
        for name in os.listdir(MOUNT):
            src = os.path.join(MOUNT, name)
            dst = os.path.join(backup, name)
            if os.path.exists(dst):
                dst = f'{dst}.{int(time.time())}'
            shutil.move(src, dst)
            moved.append((src, dst))
        print('moved non-Drive leftovers to', backup, ':', moved)
        drive.mount(MOUNT, force_remount=True)

mount_drive()
assert os.path.isdir(os.path.join(MOUNT, 'MyDrive')), 'Drive mount failed: MyDrive is missing'
OUT = os.path.join(MOUNT, 'MyDrive', 'nanlmv3_v3')
os.makedirs(OUT, exist_ok=True)
os.environ['OUT'] = OUT
print('OUT =', OUT)


## 3. Check for an existing checkpoint

`best.pt` is the preferred resume anchor. If it is absent, `last.pt`
can recover an interrupted run; if both are absent, the run starts fresh.


In [ ]:
import os
import pathlib
import time

OUT = pathlib.Path(os.environ['OUT'])
CKPT = OUT / 'best.pt'
print('output directory:', OUT)
print('best.pt exists:', CKPT.exists())
if CKPT.exists():
    stat = CKPT.stat()
    print('size_mb:', round(stat.st_size / 1e6, 1))
    print('mtime:', time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(stat.st_mtime)))
for name in ('last.pt', 'result.json', 'history.json', 'config.json', 'final.pt', 'ema.pt'):
    print(f'{name}: {(OUT / name).exists()}')


## 4. Corpus build (when the trainer takes `--corpus`)

When `z3/train_v3.py --help` advertises `--corpus` / `--data`, this cell
downloads the four releases, verifies the three published SHA256SUMS,
extracts them under `/content/z3_build`, and builds a 200 MB mixed sample
(up to 50 MB per release, then a chunk-disjoint validation tail). Otherwise
it leaves corpus handling to the trainer.

Nothing is written to Drive in this cell.


In [ ]:
import hashlib
import pathlib
import re
import subprocess
import sys
import tarfile
import urllib.request

EXP = pathlib.Path('/content/nanlm_experimental')
TRAIN = EXP / 'z3' / 'train_v3.py'
BUILD = pathlib.Path('/content/z3_build')

def trainer_help(script):
    proc = subprocess.run(
        [sys.executable, str(script), '--help'],
        text=True, capture_output=True,
    )
    if proc.returncode != 0:
        print(proc.stdout)
        print(proc.stderr)
        raise SystemExit('z3/train_v3.py --help failed')
    return proc.stdout + proc.stderr

HELP = trainer_help(TRAIN)
FLAGS = set(re.findall(r'--[A-Za-z0-9][A-Za-z0-9_-]*', HELP))
print('trainer flags:', ' '.join(sorted(FLAGS)))
CORPUS_FLAG = next(
    (flag for flag in (
        '--corpus', '--corpus-file', '--train-corpus', '--train-data', '--data'
    ) if flag in FLAGS),
    None,
)
print('corpus path flag:', CORPUS_FLAG or 'none; trainer builds its own mixed corpus')

CORPUS_ARGS = []

def download(url, dest):
    if dest.exists() and dest.stat().st_size > 0:
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    request = urllib.request.Request(url, headers={'User-Agent': 'nanlmv3-colab'})
    print('downloading', url)
    with urllib.request.urlopen(request, timeout=600) as response, dest.open('wb') as handle:
        while True:
            chunk = response.read(1 << 20)
            if not chunk:
                break
            handle.write(chunk)

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()

def extract(archive, dest, mode):
    dest.mkdir(parents=True, exist_ok=True)
    if any(dest.rglob('*.txt')):
        return
    print('extracting', archive.name, '->', dest)
    if mode == 'zst':
        import zstandard
        dctx = zstandard.ZstdDecompressor()
        with archive.open('rb') as fh, dctx.stream_reader(fh) as reader:
            with tarfile.open(fileobj=reader, mode='r|') as tf:
                try:
                    tf.extractall(dest, filter='data')
                except TypeError:
                    tf.extractall(dest)
    else:
        with tarfile.open(archive, mode='r:gz') as tf:
            try:
                tf.extractall(dest, filter='data')
            except TypeError:
                tf.extractall(dest)

if CORPUS_FLAG:
    release_dir = BUILD / 'release'
    release_dir.mkdir(parents=True, exist_ok=True)
    releases = [
        ('corpora-paragraphs-0010', 'paragraphs-0010'),
        ('corpora-short-stories-0004', 'short_stories-0004'),
        ('corpora-sentences-0010', 'sentences-0010'),
    ]
    sources = []
    for tag, prefix in releases:
        base = f'https://github.com/shemshallah/NaNROUTER/releases/download/{tag}'
        archive = release_dir / f'{prefix}.tar.zst'
        sums = release_dir / f'{prefix}.SHA256SUMS'
        download(f'{base}/{prefix}.tar.zst', archive)
        download(f'{base}/{prefix}.SHA256SUMS', sums)
        expected = sums.read_text().split()[0]
        actual = sha256(archive)
        assert actual == expected, f'SHA256 mismatch for {prefix}: {actual} != {expected}'
        dest = BUILD / f'corpus_{prefix}'
        extract(archive, dest, 'zst')
        sources.append((prefix, dest))

    gz = release_dir / 'hw5-gutenberg-c256-g1.tar.gz'
    download(
        'https://github.com/shemshallah/NaNROUTER/releases/download/'
        'hw5-gutenberg-c256-g1/hw5-gutenberg-c256-g1.tar.gz',
        gz,
    )
    gdest = BUILD / 'corpus_gutenberg'
    extract(gz, gdest, 'gz')
    sources.append(('hw5-gutenberg-c256-g1', gdest))

    prepare = pathlib.Path('/content/NaNROUTER/nanlmv3/prepare_corpus.py')
    per_source = 50_000_000
    parts = []
    for name, source in sources:
        part = BUILD / f'mix_{name}.txt'
        if not part.exists() or part.stat().st_size == 0:
            subprocess.run(
                [
                    sys.executable, str(prepare),
                    '--source', str(source),
                    '--bytes', str(per_source),
                    '--output', str(part),
                    '--manifest', str(BUILD / f'mix_{name}.json'),
                ],
                check=True,
            )
        parts.append(part)

    mixed = BUILD / 'mixed_200m.txt'
    if not mixed.exists() or mixed.stat().st_size == 0:
        with mixed.open('wb') as out:
            for part in parts:
                out.write(part.read_bytes())

    sample = BUILD / 'model_mixed_200m.txt'
    sample_bytes = min(200_000_000, mixed.stat().st_size)
    if not sample.exists() or sample.stat().st_size == 0:
        sampler = pathlib.Path('/content/NaNROUTER/nanlmv3/sample_corpus.py')
        subprocess.run(
            [
                sys.executable, str(sampler),
                '--source', str(mixed),
                '--output', str(sample),
                '--bytes', str(sample_bytes),
                '--chunks', '20',
                '--val-chunks', '1',
                '--manifest', str(BUILD / 'model_mixed_200m.json'),
            ],
            check=True,
        )

    CORPUS_ARGS = [CORPUS_FLAG, str(sample)]
    print('corpus:', sample, sample.stat().st_size, 'bytes')
else:
    print('No corpus-path flag. train_v3.py must fetch and mix the four releases itself.')


## 5. Run for 30 minutes

The command is assembled from the flags the trainer actually advertises.
`--resume-best` is preferred. The current `train_v3.py` does not define it,
so the cell falls back to `--init-from $OUT/best.pt` when `best.pt` exists,
or `--resume last` when only `last.pt` exists.


In [ ]:
import os
import pathlib
import re
import shlex
import subprocess
import sys

EXP = pathlib.Path('/content/nanlm_experimental')
TRAIN = EXP / 'z3' / 'train_v3.py'
OUT = pathlib.Path(os.environ['OUT'])
CKPT = OUT / 'best.pt'

proc = subprocess.run(
    [sys.executable, str(TRAIN), '--help'],
    text=True, capture_output=True,
)
if proc.returncode != 0:
    print(proc.stdout)
    print(proc.stderr)
    raise SystemExit('z3/train_v3.py --help failed')
FLAGS = set(re.findall(r'--[A-Za-z0-9][A-Za-z0-9_-]*', proc.stdout + proc.stderr))
print('trainer flags:', ' '.join(sorted(FLAGS)))

cmd = [sys.executable, str(TRAIN)]

def add_first(candidates, value=None):
    for flag in candidates:
        if flag in FLAGS:
            cmd.append(flag)
            if value is not None:
                cmd.append(str(value))
            return flag
    return None

out_flag = add_first(['--out', '--output-dir', '--output', '--out-dir'], str(OUT))
if out_flag is None:
    raise SystemExit('train_v3.py has no output-directory flag (--out/--output-dir)')

minutes = add_first(['--minutes', '--max-minutes', '--duration-minutes'], 30)
if minutes is None:
    seconds = add_first(['--max-seconds', '--seconds', '--time-budget'], 1800)
    if seconds is None:
        hours = add_first(['--hours', '--max-hours'], 0.5)
        if hours is None:
            print('WARNING: no time-budget flag detected; relying on the trainer default (must be 30 minutes)')

config = [
    (['--ctx', '--context'], 1024),
    (['--d-model', '--d_model', '--dim'], 256),
    (['--layers', '--n-layers', '--n_layers', '--num-layers'], 8),
    (['--heads', '--n-heads', '--n_heads', '--num-heads'], 8),
    (['--swiglu', '--ffn-hidden', '--ffn_hidden'], 1024),
    (['--expert-hidden', '--expert_hidden'], 1024),
    (['--moe-layers', '--moe_layers'], 4),
    (['--experts', '--n-experts', '--n_experts', '--num-experts'], 32),
    (['--topk', '--top-k', '--top_k'], 2),
    (['--shared-experts', '--shared_experts'], 2),
    (['--mtp', '--mtp-tokens', '--mtp_tokens'], 2),
    (['--batch-size', '--batch_size', '--batch', '--micro-batch'], 4),
]
applied = {}
for aliases, value in config:
    flag = add_first(aliases, value)
    if flag:
        applied[flag] = value
print('config flags applied:', applied)

if '--steps' in FLAGS:
    cmd += ['--steps', '20000']
if '--eval-every' in FLAGS:
    cmd += ['--eval-every', '250']
if '--log-every' in FLAGS:
    cmd += ['--log-every', '50']

for flag in ('--amp', '--fp16'):
    if flag in FLAGS:
        cmd.append(flag)
        break
if '--device' in FLAGS:
    cmd += ['--device', 'cuda']

BPE = EXP / 'z1' / 'build' / 'bpe32k_new.json'
if '--bpe-model' in FLAGS and BPE.exists():
    cmd += ['--bpe-model', str(BPE)]
    if '--tokenizer' in FLAGS:
        cmd += ['--tokenizer', 'bpe']
    print('tokenizer: bpe', BPE)
elif '--tokenizer' in FLAGS:
    cmd += ['--tokenizer', 'bytes']
    print('tokenizer: bytes (32k BPE file not found)')

CORPUS_ARGS = globals().get('CORPUS_ARGS', [])
if '--corpus' in FLAGS and not CORPUS_ARGS:
    raise SystemExit('train_v3.py requires --corpus; run the corpus cell above first')
if CORPUS_ARGS and not pathlib.Path(CORPUS_ARGS[-1]).exists():
    raise SystemExit(f'corpus file missing: {CORPUS_ARGS[-1]}')
if CORPUS_ARGS:
    cmd += CORPUS_ARGS
    print('corpus args:', CORPUS_ARGS)
else:
    print('corpus: trainer default (mixed NaNROUTER + Gutenberg)')

if '--resume-best' in FLAGS:
    cmd.append('--resume-best')
    print('resume: --resume-best')
elif CKPT.exists():
    init_flag = add_first(['--init-from', '--checkpoint', '--checkpoint-path'], str(CKPT))
    if init_flag:
        print('resume:', init_flag, CKPT)
        if '--eval-before' in FLAGS:
            cmd.append('--eval-before')
    else:
        print('WARNING: best.pt exists but the trainer has no resume/init flag; starting fresh')
elif (OUT / 'last.pt').exists() and '--resume' in FLAGS:
    cmd += ['--resume', 'last']
    print('resume: --resume last (best.pt is absent)')
else:
    print('resume: none (no best.pt or last.pt)')

print()
print('running:', shlex.join(cmd))
subprocess.run(cmd, check=True, env=dict(os.environ, PYTHONUNBUFFERED='1'))
print('training process finished')


## 6. Show result


In [ ]:
import json
import os
import pathlib

OUT = pathlib.Path(os.environ['OUT'])
candidates = [OUT / 'result.json', OUT / 'metrics.json', OUT / 'summary.json']
result = next((path for path in candidates if path.exists()), None)
if result is None:
    print('No result JSON found in', OUT)
    print('Artifacts present:')
    for path in sorted(OUT.iterdir()):
        if path.is_file():
            print(f'  {path.name}: {path.stat().st_size / 1e6:.1f} MB')
    raise SystemExit('training did not write a result JSON; inspect the log above')
print('result file:', result)
print(json.dumps(json.loads(result.read_text()), indent=2, sort_keys=True))
print()
print('Artifacts:')
for path in sorted(OUT.iterdir()):
    if path.is_file():
        print(f'  {path.name}: {path.stat().st_size / 1e6:.1f} MB')


## Resume and restart

The next run automatically prefers `--resume-best`. The current
`train_v3.py` does not define that flag, so the run cell uses
`--init-from $OUT/best.pt` when `best.pt` exists. That reloads weights but
starts a fresh optimizer and step counter. If only `last.pt` exists, the
cell uses `--resume last`, which restores optimizer, EMA, data RNG, and
step state. Because `--init-from` does not seed the trainer's best
tracker, the first evaluation writes a new `best.pt`; copy the old one
aside if you need to preserve it. There is nothing else to do after a
disconnect.

To force a fresh run, rename both checkpoints without deleting them:

```python
# !mv "$OUT/best.pt" "$OUT/best.pt.bak"
# !mv "$OUT/last.pt" "$OUT/last.pt.bak"
```

Renaming only `best.pt` falls back to `--resume last` when `last.pt`
exists. If the trainer supports `--no-resume-best`, you can pass that
instead.


## 8k context caveat

`V3Config.ctx` and `rope_base` support larger contexts, and the hybrid
local/global attention is designed to make long context cheaper.
`train_v3.py` accepts `--ctx 8192` as a preset and `rope_scaling.py`
implements NTK/YaRN, so 8k is architecture-supported. But this run uses
`ctx=1024`, and **8k is not validated here**. A T4 (16 GB) cannot hold an
8k run of this model; 8k needs an A100 40/80 GB, a small batch, gradient
checkpointing, YaRN/NTK scaling, and a long-context evaluation. This
notebook makes no 8k throughput or quality claim.
